In [ ]:
!pip install rdkit-pypi -q
# !pip install pybel -q
# !pip install PyBioMed -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
working_directory = "/content/drive/MyDrive/mtech_project/"
os.chdir(working_directory)
print("current working directory is: ", os.getcwd())

ValueError: mount failed

In [ ]:
!pip uninstall numpy rdkit-pypi -y
!pip install numpy==1.23.5 rdkit-pypi

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.4/29.4 MB 10.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
albumentations 2.0.5 requires numpy>=1.24.4, but you have numpy 1.23.5 which is incompatible.
jax 0.5.2 requires numpy>=1.25, but you have numpy 1.23.5 which is incompatible.
xarray 2025.1.2 requires numpy>=1.24, but you have numpy 1.23.5 which is incompatible.
tensorflow 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 1.23.5 which is incompatible.
jaxlib 0.5.1 requires numpy>=1.25, but you have numpy 1.23.5 which is incompatible.
treescope 0.1.9 requires numpy>=1.25.2, but you have numpy 1.23.5 which is incompatible.
albucore 0.0.23 requires num

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.Chem import Descriptors
import torch
from tqdm import tqdm
# from transformers import BertModel, BertTokenizer
# from PyBioMed.PyProtein import CTD
import warnings
warnings.filterwarnings('ignore')

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
working_directory = "/content/drive/MyDrive/mtech_project/"
os.chdir(working_directory)
print("current working directory is: ", os.getcwd())

In [ ]:
data_dir = os.path.join(os.getcwd(), 'data')
src_dir = os.path.join(os.getcwd(), 'src')
fig_dir = os.path.join(data_dir, 'figures')
checkpoint_dir = os.path.join(os.getcwd(), 'checkpoints')
dirs_to_make = [
    data_dir,
    os.path.join(data_dir, 'raw'),
    os.path.join(data_dir, 'processed'),
    os.path.join(data_dir, 'train'),
    os.path.join(data_dir, 'val'),
    os.path.join(data_dir, 'test'),
    src_dir,
    fig_dir,
    checkpoint_dir,
]
for d in dirs_to_make:
    if not os.path.exists(d):
        os.makedirs(d)

In [ ]:
train_dir = os.path.join(data_dir, 'train')
val_dir = os.path.join(data_dir, 'val')
test_dir = os.path.join(data_dir, 'test')

In [ ]:
selected_columns = ['Smiles', 'Cell Type', 'POI Sequence', 'E3 Ligase Sequence', 'Activity']

In [ ]:
def load_data(directory, columns, prefix):
    data = []
    for filename in os.listdir(directory):
        if filename.startswith(prefix) and filename.endswith('.csv'):
            file_path = os.path.join(directory, filename)
            df = pd.read_csv(file_path, usecols=columns)
            data.append(df)
    if data:
        return pd.concat(data, ignore_index=True)
    else:
        return pd.DataFrame(columns=columns)

In [ ]:
train_data = load_data(train_dir, selected_columns, 'scaffold_train_data')
val_data = load_data(val_dir, selected_columns, 'scaffold_val_data')
test_data = load_data(test_dir, selected_columns, 'scaffold_test_data')

In [ ]:
print("Train Data:")
print(train_data.shape)
print("\nValidation Data:")
print(val_data.shape)
print("\nTest Data:")
print(test_data.shape)

In [ ]:
train_data.head()

Calculating Descriptors and fingerprints for scaffold splits

These descriptors have been selected after applying Man-Whitney U-Test and using adjusted p-values using the Benjamini-Hochberg procedure

In [ ]:
descriptor_names = ['SMR_VSA9', 'fr_NH0', 'fr_piperzine',
       'NumRotatableBonds', 'SlogP_VSA3', 'BCUT2D_MWHI', 'EState_VSA10',
       'PEOE_VSA14', 'fr_Al_OH_noTert', 'NumHDonors', 'SlogP_VSA8', 'Kappa2',
       'NumSaturatedRings', 'BCUT2D_CHGLO', 'Kappa3', 'SPS', 'PEOE_VSA3',
       'qed', 'PEOE_VSA1', 'BCUT2D_MRHI', 'fr_Al_OH', 'SMR_VSA1',
       'NumSaturatedHeterocycles', 'BalabanJ', 'VSA_EState3',
       'NumAliphaticRings', 'EState_VSA2', 'NumAliphaticHeterocycles',
       'SlogP_VSA10', 'SMR_VSA6', 'fr_sulfonamd', 'SlogP_VSA7', 'EState_VSA8',
       'fr_piperdine', 'NHOHCount', 'fr_aniline', 'fr_COO2', 'fr_Al_COO',
       'fr_COO']

In [ ]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.Chem import Descriptors

# Define directories
data_dirs = {
    'train': os.path.join(os.getcwd(), 'data', 'train'),
    'val': os.path.join(os.getcwd(), 'data', 'val'),
    'test': os.path.join(os.getcwd(), 'data', 'test')
}

calculator = MoleculeDescriptors.MolecularDescriptorCalculator(descriptor_names)

# Function to compute molecular descriptors
def compute_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        return calculator.CalcDescriptors(mol)
    else:
        return [None] * len(descriptor_names)

# Function to process datasets and save descriptors
def compute_and_save_descriptors(data_dirs):
    for split, data_dir in data_dirs.items():
        for file in os.listdir(data_dir):
            if file.startswith('scaffold') and file.endswith('.csv'):
                # Load the CSV file
                file_path = os.path.join(data_dir, file)
                df = pd.read_csv(file_path)

                # Compute descriptors
                if 'Smiles' in df.columns:
                    descriptor_values = df['Smiles'].apply(compute_descriptors)
                    descriptor_df = pd.DataFrame(descriptor_values.tolist(), columns=descriptor_names)

                    # Save to new CSV file
                    output_file_path = os.path.join(data_dir, f'scaffold_{split}_descriptors.csv')
                    descriptor_df.to_csv(output_file_path, index=False)
                    print(f'Saved descriptors to {output_file_path}')
                else:
                    print(f'Skipping {file}: No Smiles column found')

# Run the function to compute and save descriptors
compute_and_save_descriptors(data_dirs)


In [ ]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.Chem import Descriptors

# Define directories
data_dirs = {
    'train': os.path.join(os.getcwd(), 'data', 'train'),
    'val': os.path.join(os.getcwd(), 'data', 'val'),
    'test': os.path.join(os.getcwd(), 'data', 'test')
}

calculator = MoleculeDescriptors.MolecularDescriptorCalculator(descriptor_names)

# Function to compute molecular descriptors
def compute_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        return calculator.CalcDescriptors(mol)
    else:
        return [None] * len(descriptor_names)

# Function to process datasets and save descriptors
def compute_and_save_descriptors(data_dirs):
    for split, data_dir in data_dirs.items():
        for file in os.listdir(data_dir):
            if file.startswith('scaffold') and file.endswith('.csv'):
                # Load the CSV file
                file_path = os.path.join(data_dir, file)
                df = pd.read_csv(file_path)

                # Compute descriptors
                if 'Smiles' in df.columns:
                    descriptor_values = df['Smiles'].apply(compute_descriptors)
                    descriptor_df = pd.DataFrame(descriptor_values.tolist(), columns=descriptor_names)

                    # Save to new CSV file
                    output_file_path = os.path.join(data_dir, f'scaffold_{split}_descriptors.csv')
                    descriptor_df.to_csv(output_file_path, index=False)
                    print(f'Saved descriptors to {output_file_path}')
                else:
                    print(f'Skipping {file}: No Smiles column found')

# Run the function to compute and save descriptors
compute_and_save_descriptors(data_dirs)

# Load test descriptors
test_descriptors_path = os.path.join(os.getcwd(), 'data', 'test', 'scaffold_test_descriptors.csv')
descriptor_df_test = pd.read_csv(test_descriptors_path)

# Now you can use descriptor_df_test
print(descriptor_df_test.head())

In [ ]:
descriptor_df_test

Fingerprints

In [ ]:
# Define directories
data_dirs = {
    'train': os.path.join(os.getcwd(), 'data', 'train'),
    'val': os.path.join(os.getcwd(), 'data', 'val'),
    'test': os.path.join(os.getcwd(), 'data', 'test')
}

In [ ]:
# Compute Morgan fingerprints
def compute_morgan_fingerprints(smiles_list, radius=4, n_bits=2048):
    fingerprints_list = []
    for smiles in smiles_list:
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            fingerprint = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
            fingerprints_list.append(fingerprint.ToBitString())
        else:
            fingerprints_list.append('0' * n_bits)  # Append a zeroed vector if the molecule is invalid
    return fingerprints_list

# Process each dataset
for split, data_dir in data_dirs.items():
    for file in os.listdir(data_dir):
        if file.startswith('scaffold') and file.endswith('.csv'):
            file_path = os.path.join(data_dir, file)
            df = pd.read_csv(file_path)

            # Compute Morgan fingerprints
            if 'Smiles' in df.columns:
                fingerprints = compute_morgan_fingerprints(df['Smiles'])

                fingerprints_int = [[int(bit) for bit in fp] for fp in fingerprints]

                fingerprints_df = pd.DataFrame(fingerprints_int, columns=[f'FP_{i}' for i in range(len(fingerprints_int[0]))])

                output_file_path = os.path.join(data_dir, f'scaffold_{split}_fingerprints.csv')
                fingerprints_df.to_csv(output_file_path, index=False)
                print(f'Saved fingerprints to {output_file_path}')
            else:
                print(f'Skipping {file}: No Smiles column found')

print("Morgan fingerprints calculation and saving complete.")

In [ ]:
# check the shape of saved fingerprint data
def check_fingerprint_shapes(data_dirs):
    for split, data_dir in data_dirs.items():
        for file in os.listdir(data_dir):
            if file.startswith('scaffold') and file.endswith('_fingerprints.csv'):
                file_path = os.path.join(data_dir, file)
                df = pd.read_csv(file_path)

                print(f'Shape of {file}: {df.shape}')

check_fingerprint_shapes(data_dirs)

Descriptors for POI sequences

In [ ]:
train_data.head(3)

In [ ]:
poi_seq = train_data['POI Sequence'].to_list()

In [ ]:
poi_seq

Calc POI seq Descriptors

In [ ]:
!pip install PyBioMed

import os
import pandas as pd
from rdkit import Chem
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.Chem import Descriptors
# Import CTD
from PyBioMed.PyProtein import CTD
def compute_protein_descriptors(data_dirs):
    for split, data_dir in data_dirs.items():
        for file in os.listdir(data_dir):
            if file.startswith('scaffold') and file.startswith('scaffold') and file.endswith('.csv'):
                file_path = os.path.join(data_dir, file)
                df = pd.read_csv(file_path)

                # Compute descriptors
                if 'POI Sequence' in df.columns:
                    poi_seq = df['POI Sequence'].to_list()
                    data = []

                    for protein in poi_seq:
                        protein_descriptor = CTD.CalculateC(protein)
                        data.append(protein_descriptor)

                    descriptors_df = pd.DataFrame(data)

                    output_file_path = os.path.join(data_dir, f'scaffold_{split}_protein_descriptors.csv')
                    descriptors_df.to_csv(output_file_path, index=False)
                    print(f'Saved protein descriptors to {output_file_path}')
                else:
                    print(f'Skipping {file}: No POI Sequence column found')

compute_protein_descriptors(data_dirs)


In [ ]:
# Function to check the shape of saved fingerprint data
def check_fingerprint_shapes(data_dirs):
    for split, data_dir in data_dirs.items():
        for file in os.listdir(data_dir):
            if file.startswith('scaffold') and file.endswith('_protein_descriptors.csv'):
                file_path = os.path.join(data_dir, file)
                df = pd.read_csv(file_path)

                print(f'Shape of {file}: {df.shape}')

check_fingerprint_shapes(data_dirs)

In [ ]:
train_data.columns

In [ ]:
# @title **Calc E3 Descriptors**

def compute_E3_descriptors(data_dirs):
    for split, data_dir in data_dirs.items():
        for file in os.listdir(data_dir):
            if file.startswith('scaffold') and file.endswith('.csv'):
                file_path = os.path.join(data_dir, file)
                df = pd.read_csv(file_path)

                if 'E3 Ligase Sequence' in df.columns:
                    poi_seq = df['E3 Ligase Sequence'].to_list()
                    data = []

                    for protein in poi_seq:
                        protein_descriptor = CTD.CalculateC(protein)
                        data.append(protein_descriptor)

                    descriptors_df = pd.DataFrame(data)

                    output_file_path = os.path.join(data_dir, f'scaffold_{split}_e3_descriptors.csv')
                    descriptors_df.to_csv(output_file_path, index=False)
                    print(f'Saved E3 ligase descriptors to {output_file_path}')
                else:
                    print(f'Skipping {file}: No E3 ligase Sequence column found')

compute_E3_descriptors(data_dirs)


In [ ]:
# Function to check the shape of saved fingerprint data
def check_fingerprint_shapes(data_dirs):
    for split, data_dir in data_dirs.items():
        for file in os.listdir(data_dir):
            if file.startswith('scaffold') and file.endswith('_e3_descriptors.csv'):
                file_path = os.path.join(data_dir, file)
                df = pd.read_csv(file_path)

                print(f'Shape of {file}: {df.shape}')

check_fingerprint_shapes(data_dirs)

Now we need to calculate Embeddings for POI Sequence and E3 ligase

Embeddings for POI Sequence



In [ ]:
poi_seq = train_data['POI Sequence'].to_list()
poi_seq

In [ ]:
!pip install numpy --upgrade --ignore-installed

In [ ]:
# Load ProtBERT Model
print("Load ProtBERT Model...")
from transformers import BertTokenizer, BertModel # Import BertTokenizer and BertModel
tokenizer = BertTokenizer.from_pretrained("Rostlab/prot_bert", do_lower_case=False)
model = BertModel.from_pretrained("Rostlab/prot_bert").to('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
!pip install importlib_metadata
# Install necessary packages with version specification
!pip install transformers==4.28.1 torch==1.13.1 --quiet

# Load ProtBERT Model
print("Load ProtBERT Model...")
from transformers import BertTokenizer, BertModel  # Import BertTokenizer and BertModel

# Initialize the tokenizer and model with specified model name
tokenizer = BertTokenizer.from_pretrained("Rostlab/prot_bert", do_lower_case=False)
model = BertModel.from_pretrained("Rostlab/prot_bert")

# Move the model to the appropriate device (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
import transformers
print(transformers.__version__)

In [ ]:


!pip install rdkit-pypi -q
# !pip install pybel -q
# !pip install PyBioMed -q

from google.colab import drive
drive.mount('/content/drive')
working_directory = "/content/drive/MyDrive/mtech_project/"
os.chdir(working_directory)
print("current working directory is: ", os.getcwd())

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.Chem import Descriptors
import torch
from tqdm import tqdm
# from transformers import BertModel, BertTokenizer
# from PyBioMed.PyProtein import CTD
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')
working_directory = "/content/drive/MyDrive/mtech_project/"
os.chdir(working_directory)
print("current working directory is: ", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
current working directory is:  /content/drive/MyDrive/mtech_project

data_dir = os.path.join(os.getcwd(), 'data')
src_dir = os.path.join(os.getcwd(), 'src')
fig_dir = os.path.join(data_dir, 'figures')
checkpoint_dir = os.path.join(os.getcwd(), 'checkpoints')
dirs_to_make = [
    data_dir,
    os.path.join(data_dir, 'raw'),
    os.path.join(data_dir, 'processed'),
    os.path.join(data_dir, 'train'),
    os.path.join(data_dir, 'val'),
    os.path.join(data_dir, 'test'),
    src_dir,
    fig_dir,
    checkpoint_dir,
]
for d in dirs_to_make:
    if not os.path.exists(d):
        os.makedirs(d)

train_dir = os.path.join(data_dir, 'train')
val_dir = os.path.join(data_dir, 'val')
test_dir = os.path.join(data_dir, 'test')

selected_columns = ['Smiles', 'Cell Type', 'POI Sequence', 'E3 Ligase Sequence', 'Activity']

def load_data(directory, columns, prefix):
    data = []
    for filename in os.listdir(directory):
        if filename.startswith(prefix) and filename.endswith('.csv'):
            file_path = os.path.join(directory, filename)
            df = pd.read_csv(file_path, usecols=columns)
            data.append(df)
    if data:
        return pd.concat(data, ignore_index=True)
    else:
        return pd.DataFrame(columns=columns)

train_data = load_data(train_dir, selected_columns, 'scaffold_train_data')
val_data = load_data(val_dir, selected_columns, 'scaffold_val_data')
test_data = load_data(test_dir, selected_columns, 'scaffold_test_data')

print("Train Data:")
print(train_data.shape)
print("\nValidation Data:")
print(val_data.shape)
print("\nTest Data:")
print(test_data.shape)

Train Data:
(396, 5)

Validation Data:
(77, 5)

Test Data:
(31, 5)

Calculating Descriptors and fingerprints for scaffold splits

These descriptors have been selected after applying Man-Whitney U-Test and using adjusted p-values using the Benjamini-Hochberg procedure

descriptor_names = ['SMR_VSA9', 'fr_NH0', 'fr_piperzine',
       'NumRotatableBonds', 'SlogP_VSA3', 'BCUT2D_MWHI', 'EState_VSA10',
       'PEOE_VSA14', 'fr_Al_OH_noTert', 'NumHDonors', 'SlogP_VSA8', 'Kappa2',
       'NumSaturatedRings', 'BCUT2D_CHGLO', 'Kappa3', 'SPS', 'PEOE_VSA3',
       'qed', 'PEOE_VSA1', 'BCUT2D_MRHI', 'fr_Al_OH', 'SMR_VSA1',
       'NumSaturatedHeterocycles', 'BalabanJ', 'VSA_EState3',
       'NumAliphaticRings', 'EState_VSA2', 'NumAliphaticHeterocycles',
       'SlogP_VSA10', 'SMR_VSA6', 'fr_sulfonamd', 'SlogP_VSA7', 'EState_VSA8',
       'fr_piperdine', 'NHOHCount', 'fr_aniline', 'fr_COO2', 'fr_Al_COO',
       'fr_COO']

import os
import pandas as pd
from rdkit import Chem
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.Chem import Descriptors

# Define directories
data_dirs = {
    'train': os.path.join(os.getcwd(), 'data', 'train'),
    'val': os.path.join(os.getcwd(), 'data', 'val'),
    'test': os.path.join(os.getcwd(), 'data', 'test')
}

calculator = MoleculeDescriptors.MolecularDescriptorCalculator(descriptor_names)

# Function to compute molecular descriptors
def compute_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        return calculator.CalcDescriptors(mol)
    else:
        return [None] * len(descriptor_names)

# Function to process datasets and save descriptors
def compute_and_save_descriptors(data_dirs):
    for split, data_dir in data_dirs.items():
        for file in os.listdir(data_dir):
            if file.startswith('scaffold') and file.endswith('.csv'):
                # Load the CSV file
                file_path = os.path.join(data_dir, file)
                df = pd.read_csv(file_path)

                # Compute descriptors
                if 'Smiles' in df.columns:
                    descriptor_values = df['Smiles'].apply(compute_descriptors)
                    descriptor_df = pd.DataFrame(descriptor_values.tolist(), columns=descriptor_names)

                    # Save to new CSV file
                    output_file_path = os.path.join(data_dir, f'scaffold_{split}_descriptors.csv')
                    descriptor_df.to_csv(output_file_path, index=False)
                    print(f'Saved descriptors to {output_file_path}')
                else:
                    print(f'Skipping {file}: No Smiles column found')

# Run the function to compute and save descriptors
compute_and_save_descriptors(data_dirs)

Saved descriptors to /content/drive/MyDrive/mtech_project/data/train/scaffold_train_descriptors.csv
Skipping scaffold_train_descriptors.csv: No Smiles column found
Skipping scaffold_train_fingerprints.csv: No Smiles column found
Skipping scaffold_train_protein_descriptors.csv: No Smiles column found
Skipping scaffold_train_e3_descriptors.csv: No Smiles column found
Saved descriptors to /content/drive/MyDrive/mtech_project/data/val/scaffold_val_descriptors.csv
Skipping scaffold_val_descriptors.csv: No Smiles column found
Skipping scaffold_val_fingerprints.csv: No Smiles column found
Skipping scaffold_val_protein_descriptors.csv: No Smiles column found
Skipping scaffold_val_e3_descriptors.csv: No Smiles column found
Saved descriptors to /content/drive/MyDrive/mtech_project/data/test/scaffold_test_descriptors.csv
Skipping scaffold_test_descriptors.csv: No Smiles column found
Skipping scaffold_test_fingerprints.csv: No Smiles column found
Skipping scaffold_test_protein_descriptors.csv: No Smiles column found
Skipping scaffold_test_e3_descriptors.csv: No Smiles column found

Saved descriptors to /content/drive/MyDrive/mtech_project/data/train/scaffold_train_descriptors.csv
Skipping scaffold_train_descriptors.csv: No Smiles column found
Skipping scaffold_train_fingerprints.csv: No Smiles column found
Skipping scaffold_train_protein_descriptors.csv: No Smiles column found
Skipping scaffold_train_e3_descriptors.csv: No Smiles column found
Saved descriptors to /content/drive/MyDrive/mtech_project/data/val/scaffold_val_descriptors.csv
Skipping scaffold_val_descriptors.csv: No Smiles column found
Skipping scaffold_val_fingerprints.csv: No Smiles column found
Skipping scaffold_val_protein_descriptors.csv: No Smiles column found
Skipping scaffold_val_e3_descriptors.csv: No Smiles column found
Saved descriptors to /content/drive/MyDrive/mtech_project/data/test/scaffold_test_descriptors.csv
Skipping scaffold_test_descriptors.csv: No Smiles column found
Skipping scaffold_test_fingerprints.csv: No Smiles column found
Skipping scaffold_test_protein_descriptors.csv: No Smiles column found
Skipping scaffold_test_e3_descriptors.csv: No Smiles column found
    SMR_VSA9  fr_NH0  fr_piperzine  NumRotatableBonds  SlogP_VSA3  \
0   5.749512       6             0                 15   23.057568
1  11.818733       5             1                  9    9.589074
2  16.814289       7             0                 22   38.617492
3  11.818733       6             1                  7    9.589074
4  17.636596       5             0                 16   23.915012

   BCUT2D_MWHI  EState_VSA10  PEOE_VSA14  fr_Al_OH_noTert  NumHDonors  ...  \
0    79.918732     28.767223         0.0                0           2  ...
1    35.495694     23.972686         0.0                0           2  ...
2    32.233232     32.390483         0.0                0           3  ...
3    35.495693     23.972686         0.0                0           1  ...
4    16.485474     38.823652         0.0                0           5  ...

    SMR_VSA6  fr_sulfonamd  SlogP_VSA7  EState_VSA8  fr_piperdine  NHOHCount  \
0  64.145622             0    0.000000    84.907897             2          2
1  55.613114             0    5.022633    31.402528             2          2
2  87.014880             1    0.000000    25.323629             2          3
3  62.157871             0    5.022633    38.218473             3          1
4  38.226947             0    0.000000    42.621121             1          5

   fr_aniline  fr_COO2  fr_Al_COO  fr_COO
0           2        0          0       0
1           2        0          0       0
2           3        0          0       0
3           2        0          0       0
4           1        0          0       0

[5 rows x 39 columns]

descriptor_df_test

Fingerprints

# Define directories
data_dirs = {
    'train': os.path.join(os.getcwd(), 'data', 'train'),
    'val': os.path.join(os.getcwd(), 'data', 'val'),
    'test': os.path.join(os.getcwd(), 'data', 'test')
}

# Compute Morgan fingerprints
def compute_morgan_fingerprints(smiles_list, radius=4, n_bits=2048):
    fingerprints_list = []
    for smiles in smiles_list:
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            fingerprint = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
            fingerprints_list.append(fingerprint.ToBitString())
        else:
            fingerprints_list.append('0' * n_bits)  # Append a zeroed vector if the molecule is invalid
    return fingerprints_list

# Process each dataset
for split, data_dir in data_dirs.items():
    for file in os.listdir(data_dir):
        if file.startswith('scaffold') and file.endswith('.csv'):
            file_path = os.path.join(data_dir, file)
            df = pd.read_csv(file_path)

            # Compute Morgan fingerprints
            if 'Smiles' in df.columns:
                fingerprints = compute_morgan_fingerprints(df['Smiles'])

                fingerprints_int = [[int(bit) for bit in fp] for fp in fingerprints]

                fingerprints_df = pd.DataFrame(fingerprints_int, columns=[f'FP_{i}' for i in range(len(fingerprints_int[0]))])

                output_file_path = os.path.join(data_dir, f'scaffold_{split}_fingerprints.csv')
                fingerprints_df.to_csv(output_file_path, index=False)
                print(f'Saved fingerprints to {output_file_path}')
            else:
                print(f'Skipping {file}: No Smiles column found')

print("Morgan fingerprints calculation and saving complete.")

Saved fingerprints to /content/drive/MyDrive/mtech_project/data/train/scaffold_train_fingerprints.csv
Skipping scaffold_train_descriptors.csv: No Smiles column found
Skipping scaffold_train_fingerprints.csv: No Smiles column found
Skipping scaffold_train_protein_descriptors.csv: No Smiles column found
Skipping scaffold_train_e3_descriptors.csv: No Smiles column found
Saved fingerprints to /content/drive/MyDrive/mtech_project/data/val/scaffold_val_fingerprints.csv
Skipping scaffold_val_descriptors.csv: No Smiles column found
Skipping scaffold_val_fingerprints.csv: No Smiles column found
Skipping scaffold_val_protein_descriptors.csv: No Smiles column found
Skipping scaffold_val_e3_descriptors.csv: No Smiles column found
Saved fingerprints to /content/drive/MyDrive/mtech_project/data/test/scaffold_test_fingerprints.csv
Skipping scaffold_test_descriptors.csv: No Smiles column found
Skipping scaffold_test_fingerprints.csv: No Smiles column found
Skipping scaffold_test_protein_descriptors.csv: No Smiles column found
Skipping scaffold_test_e3_descriptors.csv: No Smiles column found
Morgan fingerprints calculation and saving complete.

# check the shape of saved fingerprint data
def check_fingerprint_shapes(data_dirs):
    for split, data_dir in data_dirs.items():
        for file in os.listdir(data_dir):
            if file.startswith('scaffold') and file.endswith('_fingerprints.csv'):
                file_path = os.path.join(data_dir, file)
                df = pd.read_csv(file_path)

                print(f'Shape of {file}: {df.shape}')

check_fingerprint_shapes(data_dirs)

Shape of scaffold_train_fingerprints.csv: (396, 2048)
Shape of scaffold_val_fingerprints.csv: (77, 2048)
Shape of scaffold_test_fingerprints.csv: (31, 2048)

Descriptors for POI sequences

train_data.head(3)

poi_seq = train_data['POI Sequence'].to_list()

['...']

Calc POI seq Descriptors

!pip install PyBioMed

import os
import pandas as pd
from rdkit import Chem
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.Chem import Descriptors
# Import CTD
from PyBioMed.PyProtein import CTD
def compute_protein_descriptors(data_dirs):
    for split, data_dir in data_dirs.items():
        for file in os.listdir(data_dir):
            if file.startswith('scaffold') and file.startswith('scaffold') and file.endswith('.csv'):
                file_path = os.path.join(data_dir, file)
                df = pd.read_csv(file_path)

                # Compute descriptors
                if 'POI Sequence' in df.columns:
                    poi_seq = df['POI Sequence'].to_list()
                    data = []

                    for protein in poi_seq:
                        protein_descriptor = CTD.CalculateC(protein)
                        data.append(protein_descriptor)

                    descriptors_df = pd.DataFrame(data)

                    output_file_path = os.path.join(data_dir, f'scaffold_{split}_protein_descriptors.csv')
                    descriptors_df.to_csv(output_file_path, index=False)
                    print(f'Saved protein descriptors to {output_file_path}')
                else:
                    print(f'Skipping {file}: No POI Sequence column found')

compute_protein_descriptors(data_dirs)

Requirement already satisfied: PyBioMed in /usr/local/lib/python3.11/dist-packages (1.0)
Saved protein descriptors to /content/drive/MyDrive/mtech_project/data/train/scaffold_train_protein_descriptors.csv
Skipping scaffold_train_descriptors.csv: No POI Sequence column found
Skipping scaffold_train_fingerprints.csv: No POI Sequence column found
Skipping scaffold_train_protein_descriptors.csv: No POI Sequence column found
Skipping scaffold_train_e3_descriptors.csv: No POI Sequence column found
Saved protein descriptors to /content/drive/MyDrive/mtech_project/data/val/scaffold_val_protein_descriptors.csv
Skipping scaffold_val_descriptors.csv: No POI Sequence column found
Skipping scaffold_val_fingerprints.csv: No POI Sequence column found
Skipping scaffold_val_protein_descriptors.csv: No POI Sequence column found
Skipping scaffold_val_e3_descriptors.csv: No POI Sequence column found
Saved protein descriptors to /content/drive/MyDrive/mtech_project/data/test/scaffold_test_protein_descriptors.csv
Skipping scaffold_test_descriptors.csv: No POI Sequence column found
Skipping scaffold_test_fingerprints.csv: No POI Sequence column found
Skipping scaffold_test_protein_descriptors.csv: No POI Sequence column found
Skipping scaffold_test_e3_descriptors.csv: No POI Sequence column found

# Function to check the shape of saved fingerprint data
def check_fingerprint_shapes(data_dirs):
    for split, data_dir in data_dirs.items():
        for file in os.listdir(data_dir):
            if file.startswith('scaffold') and file.endswith('_protein_descriptors.csv'):
                file_path = os.path.join(data_dir, file)
                df = pd.read_csv(file_path)

                print(f'Shape of {file}: {df.shape}')

check_fingerprint_shapes(data_dirs)

Shape of scaffold_train_protein_descriptors.csv: (396, 21)
Shape of scaffold_val_protein_descriptors.csv: (77, 21)
Shape of scaffold_test_protein_descriptors.csv: (31, 21)

train_data.columns

Index(['Smiles', 'Cell Type', 'POI Sequence', 'E3 Ligase Sequence',
       'Activity'],
      dtype='object')

Calc E3 Descriptors

# @title **Calc E3 Descriptors**

def compute_E3_descriptors(data_dirs):
    for split, data_dir in data_dirs.items():
        for file in os.listdir(data_dir):
            if file.startswith('scaffold') and file.endswith('.csv'):
                file_path = os.path.join(data_dir, file)
                df = pd.read_csv(file_path)

                if 'E3 Ligase Sequence' in df.columns:
                    poi_seq = df['E3 Ligase Sequence'].to_list()
                    data = []

                    for protein in poi_seq:
                        protein_descriptor = CTD.CalculateC(protein)
                        data.append(protein_descriptor)

                    descriptors_df = pd.DataFrame(data)

                    output_file_path = os.path.join(data_dir, f'scaffold_{split}_e3_descriptors.csv')
                    descriptors_df.to_csv(output_file_path, index=False)
                    print(f'Saved E3 ligase descriptors to {output_file_path}')
                else:
                    print(f'Skipping {file}: No E3 ligase Sequence column found')

compute_E3_descriptors(data_dirs)

Saved E3 ligase descriptors to /content/drive/MyDrive/mtech_project/data/train/scaffold_train_e3_descriptors.csv
Skipping scaffold_train_descriptors.csv: No E3 ligase Sequence column found
Skipping scaffold_train_fingerprints.csv: No E3 ligase Sequence column found
Skipping scaffold_train_protein_descriptors.csv: No E3 ligase Sequence column found
Skipping scaffold_train_e3_descriptors.csv: No E3 ligase Sequence column found
Saved E3 ligase descriptors to /content/drive/MyDrive/mtech_project/data/val/scaffold_val_e3_descriptors.csv
Skipping scaffold_val_descriptors.csv: No E3 ligase Sequence column found
Skipping scaffold_val_fingerprints.csv: No E3 ligase Sequence column found
Skipping scaffold_val_protein_descriptors.csv: No E3 ligase Sequence column found
Skipping scaffold_val_e3_descriptors.csv: No E3 ligase Sequence column found
Saved E3 ligase descriptors to /content/drive/MyDrive/mtech_project/data/test/scaffold_test_e3_descriptors.csv
Skipping scaffold_test_descriptors.csv: No E3 ligase Sequence column found
Skipping scaffold_test_fingerprints.csv: No E3 ligase Sequence column found
Skipping scaffold_test_protein_descriptors.csv: No E3 ligase Sequence column found
Skipping scaffold_test_e3_descriptors.csv: No E3 ligase Sequence column found

# Function to check the shape of saved fingerprint data
def check_fingerprint_shapes(data_dirs):
    for split, data_dir in data_dirs.items():
        for file in os.listdir(data_dir):
            if file.startswith('scaffold') and file.endswith('_e3_descriptors.csv'):
                file_path = os.path.join(data_dir, file)
                df = pd.read_csv(file_path)

                print(f'Shape of {file}: {df.shape}')

check_fingerprint_shapes(data_dirs)

Shape of scaffold_train_e3_descriptors.csv: (396, 21)
Shape of scaffold_val_e3_descriptors.csv: (77, 21)
Shape of scaffold_test_e3_descriptors.csv: (31, 21)

Now we need to calculate Embeddings for POI Sequence and E3 ligase

Embeddings for POI Sequence

poi_seq = train_data['POI Sequence'].to_list()
poi_seq

['...']

!pip install numpy --upgrade --ignore-installed

Collecting numpy
  Using cached numpy-2.2.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
Using cached numpy-2.2.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.4 MB)
Installing collected packages: numpy
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.61.0 requires numpy<2.2,>=1.24, but you have numpy 2.2.4 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.2.4 which is incompatible.
distributed-ucxx-cu12 0.42.0 requires numba<0.61.0a0,>=0.59.1, but you have numba 0.61.0 which is incompatible.
tf-keras 2.18.0 requires tensorflow<2.19,>=2.18, but you have tensorflow 2.19.0 which is incompatible.
tensorflow-text 2.18.1 requires tensorflow<2.19,>=2.18.0, but you have tensorflow 2.19.0 which is incompatible.
cuml-cu12 25.2.1 requires numba<0.61.0a0,>=0.59.1, but you have numba 0.61.0 which is incompatible.
dask-cuda 25.2.0 requires numba<0.61.0a0,>=0.59.1, but you have numba 0.61.0 which is incompatible.
cudf-cu12 25.2.1 requires numba<0.61.0a0,>=0.59.1, but you have numba 0.61.0 which is incompatible.
Successfully installed numpy-2.1.3

# Load ProtBERT Model
print("Load ProtBERT Model...")
from transformers import BertTokenizer, BertModel # Import BertTokenizer and BertModel
tokenizer = BertTokenizer.from_pretrained("Rostlab/prot_bert", do_lower_case=False)
model = BertModel.from_pretrained("Rostlab/prot_bert").to('cuda' if torch.cuda.is_available() else 'cpu')

4.49.0


In [ ]:
!pip install transformers==4.28.1 torch==1.13.1 importlib_metadata --quiet # Install importlib_metadata along with transformers and torch

In [ ]:
!pip install --upgrade numpy


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 29.2 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.23.5
    Uninstalling numpy-1.23.5:
      Successfully uninstalled numpy-1.23.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.4 which is incompatible.
tensorflow 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.2.4 which is incompatible.


In [ ]:
!pip install --upgrade transformers==4.30.0  # You can try an earlier stable version
!pip install --upgrade torch==2.0.1  # Ensure that torch is also a stable version


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.6/113.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 24.2 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.1
    Uninstalling tokenizers-0.21.1:
      Successfully uninstalled tokenizers-0.21.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.49.0
    Uninstalling transformers-4.49.0:
      Successfully uninstalled transformers-4.49.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 3.4.1 requires transformers<5.0.0,>=4.41.0, but you have transformers 4.30.0 which is incompatible.


ERROR: Operation cancelled by user
^C


In [ ]:
import numpy as np
import torch
import transformers

print("Numpy version:", np.__version__)
print("Torch version:", torch.__version__)
print("Transformers version:", transformers.__version__)


Numpy version: 2.2.4
Torch version: 2.6.0+cu124
Transformers version: 4.30.0


In [ ]:
!pip uninstall numpy -y


Found existing installation: numpy 2.2.4
Uninstalling numpy-2.2.4:
  Successfully uninstalled numpy-2.2.4


In [ ]:
!pip install numpy==1.23.5


  Using cached numpy-1.23.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.3 kB)
Using cached numpy-1.23.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (17.1 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
albumentations 2.0.5 requires numpy>=1.24.4, but you have numpy 1.23.5 which is incompatible.
jax 0.5.2 requires numpy>=1.25, but you have numpy 1.23.5 which is incompatible.
xarray 2025.1.2 requires numpy>=1.24, but you have numpy 1.23.5 which is incompatible.
tensorflow 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 1.23.5 which is incompatible.
jaxlib 0.5.1 requires numpy>=1.25, but you have numpy 1.23.5 which is incompatible.
treescope 0.1.9 requires numpy>=1.25.2, but you have numpy 1.23.5 which is incompatible.
albucore 0.0.23 requires numpy>=1.24.4, but you have numpy 1.23.5 which is incompatible.
big

In [ ]:
import numpy as np
print("Numpy version:", np.__version__)


Numpy version: 1.23.5


In [ ]:
!pip install --upgrade transformers
!pip install --upgrade torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 53.7 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.13.3
    Uninstalling tokenizers-0.13.3:
      Successfully uninstalled tokenizers-0.13.3
  Attempting uninstall: transformers
    Found existing installation: transformers 4.30.0
    Uninstalling transformers-4.30.0:
      Successfully uninstalled transformers-4.30.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [ ]:
import torch
import transformers

print("Torch version:", torch.__version__)
print("Transformers version:", transformers.__version__)


Torch version: 2.6.0+cu124
Transformers version: 4.49.0


In [ ]:
import numpy as np
np.dtypes = np.dtype  #

In [ ]:
!pip install --upgrade jax jaxlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.1/105.1 MB 8.0 MB/s eta 0:00:00
  Attempting uninstall: jaxlib
    Found existing installation: jaxlib 0.5.1
    Uninstalling jaxlib-0.5.1:
      Successfully uninstalled jaxlib-0.5.1
  Attempting uninstall: jax
    Found existing installation: jax 0.5.2
    Uninstalling jax-0.5.2:
      Successfully uninstalled jax-0.5.2


In [ ]:
import torch
# Load ProtBERT Model
print("Load ProtBERT Model...")
from transformers import BertTokenizer, BertModel
tokenizer = BertTokenizer.from_pretrained("Rostlab/prot_bert", do_lower_case=False)
model = BertModel.from_pretrained("Rostlab/prot_bert").to('cuda' if torch.cuda.is_available() else 'cpu')




Load ProtBERT Model...


RuntimeError: Failed to import transformers.models.bert.modeling_bert because of the following error (look up to see its traceback):
Failed to import transformers.generation.utils because of the following error (look up to see its traceback):
type object 'numpy.dtype' has no attribute 'UInt32DType'

In [ ]:
import numpy as np
np.dtypes = np.dtype  # Assign np.dtype to np.dtypes

In [ ]:
# Get ProtBERT embeddings
def get_bert_embedding(sequence: str, len_seq_limit: int = 1200):
    sequence_w_spaces = ' '.join(list(sequence))
    encoded_input = tokenizer(
        sequence_w_spaces,
        truncation=True,
        max_length=len_seq_limit,
        padding='max_length',
        return_tensors='pt').to('cuda' if torch.cuda.is_available() else 'cpu')
    with torch.no_grad():
        output = model(**encoded_input)
    output_hidden = output['last_hidden_state'][:, 0, :].cpu().numpy()
    return output_hidden

In [ ]:
# process sequences and save embeddings
def process_sequences(data_path, save_path):
    data = pd.read_csv(data_path)
    poi_seq = data['POI Sequence'].tolist()
    embeddings_list = []
    for seq in tqdm(poi_seq):
        embedding = get_bert_embedding(sequence=seq)
        embeddings_list.append(embedding)
    embeddings_array = np.array(embeddings_list)
    np.save(save_path, embeddings_array)
    print(f"Embeddings saved to {save_path}")

In [ ]:
data_dir = os.path.join(os.getcwd(), 'data')
src_dir = os.path.join(os.getcwd(), 'src')
fig_dir = os.path.join(data_dir, 'figures')
checkpoint_dir = os.path.join(os.getcwd(), 'checkpoints')
dirs_to_make = [
    data_dir,
    os.path.join(data_dir, 'raw'),
    os.path.join(data_dir, 'processed'),
    os.path.join(data_dir, 'train'),
    os.path.join(data_dir, 'val'),
    os.path.join(data_dir, 'test'),
    src_dir,
    fig_dir,
    checkpoint_dir,
]
for d in dirs_to_make:
    if not os.path.exists(d):
        os.makedirs(d)

In [ ]:
train_dir = os.path.join(data_dir, 'train')
val_dir = os.path.join(data_dir, 'val')
test_dir = os.path.join(data_dir, 'test')

In [ ]:
import os
# Define paths for train, val, and test sets
train_data_path = os.path.join(train_dir + '/scaffold_train_data.csv')
val_data_path = os.path.join(val_dir +'/scaffold_val_data.csv')
test_data_path = os.path.join(test_dir + '/scaffold_test_data.csv')

In [ ]:
data_dir

'/content/data'

In [ ]:
# Define paths to save embeddings
train_save_path = os.path.join(data_dir, 'train', 'scaffold_protbert_train_embeddings.npy')
val_save_path = os.path.join(data_dir, 'val', 'scaffold_protbert_val_embeddings.npy')
test_save_path = os.path.join(data_dir, 'test', 'scaffold_protbert_test_embeddings.npy')

In [ ]:
import pandas as pd  # Import pandas with alias 'pd'


# process sequences and save embeddings
def process_sequences(data_path, save_path):
    data = pd.read_csv(data_path)  # Now 'pd' will be recognized
    poi_seq = data['POI Sequence'].tolist()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
working_directory = "/content/drive/MyDrive/mtech_project/"
os.chdir(working_directory)
print("current working directory is: ", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
current working directory is:  /content/drive/MyDrive/mtech_project


In [ ]:
# ipython-input-15-cd9a0395e3ab

# Define paths to save embeddings
train_save_path = os.path.join(data_dir, 'train', 'scaffold_protbert_train_embeddings.npy')
val_save_path = os.path.join(data_dir, 'val', 'scaffold_protbert_val_embeddings.npy')
test_save_path = os.path.join(data_dir, 'test', 'scaffold_protbert_test_embeddings.npy')


# Define paths to your data files using os.getcwd() to get the working directory
train_data_path = os.path.join(os.getcwd(), 'data','train', 'scaffold_train_data.csv')  # Directly in the working directory
val_data_path = os.path.join(os.getcwd(), 'data', 'val','scaffold_val_data.csv')  # Directly in the working directory
test_data_path = os.path.join(os.getcwd(), 'data','test', 'scaffold_test_data.csv')  # Directly in the working directory

NameError: name 'os' is not defined

In [ ]:
#import pandas as pd
# Process and save embeddings for train, val, and test sets
print("Processing train set...")
process_sequences(train_data_path, train_save_path)

print("Processing validation set...")
process_sequences(val_data_path, val_save_path)

print("Processing test set...")
process_sequences(test_data_path, test_save_path)

Processing train set...


NameError: name 'process_sequences' is not defined

Embeddings For E3 ligase

In [ ]:
# process sequences and save embeddings
def process_sequences(data_path, save_path):
    data = pd.read_csv(data_path)
    e3_seq = data['E3 Ligase Sequence'].tolist()
    embeddings_list = []
    for seq in tqdm(e3_seq):
        embedding = get_bert_embedding(sequence=seq)
        embeddings_list.append(embedding)
    embeddings_array = np.array(embeddings_list)
    np.save(save_path, embeddings_array)
    print(f"Embeddings saved to {save_path}")

In [ ]:
# Define paths to save embeddings
train_save_path = os.path.join(data_dir, 'train', 'scaffold_protbert_E3_train_embeddings.npy')
val_save_path = os.path.join(data_dir, 'val', 'scaffold_protbert_E3_val_embeddings.npy')
test_save_path = os.path.join(data_dir, 'test', 'scaffold_protbert_E3_test_embeddings.npy')

In [ ]:
!pip install tqdm -q # install tqdm if it's not already installed
from tqdm import tqdm # import the tqdm module

import pandas as pd
# process sequences and save embeddings
def process_sequences(data_path, save_path):
    data = pd.read_csv(data_path)
    e3_seq = data['E3 Ligase Sequence'].tolist()
    embeddings_list = []
    for seq in tqdm(e3_seq): # Wrap e3_seq with tqdm to display progress bar
        embedding = get_bert_embedding(sequence=seq)
        embeddings_list.append(embedding)
    embeddings_array = np.array(embeddings_list)
    np.save(save_path, embeddings_array)
    print(f"Embeddings saved to {save_path}")

In [ ]:
# Process and save embeddings for train, val, and test sets
print("Processing train set...")
process_sequences(train_data_path, train_save_path)

print("Processing validation set...")
process_sequences(val_data_path, val_save_path)

print("Processing test set...")
process_sequences(test_data_path, test_save_path)

Processing train set...


100%|██████████| 396/396 [2:40:51<00:00, 24.37s/it]


Embeddings saved to /content/data/train/scaffold_protbert_E3_train_embeddings.npy
Processing validation set...


100%|██████████| 77/77 [31:19<00:00, 24.41s/it]


Embeddings saved to /content/data/val/scaffold_protbert_E3_val_embeddings.npy
Processing test set...


100%|██████████| 31/31 [12:43<00:00, 24.62s/it]

Embeddings saved to /content/data/test/scaffold_protbert_E3_test_embeddings.npy


In [3]:
!cp /content/data/val/scaffold_protbert_E3_val_embeddings.npy /content/drive/MyDrive/mtech_project/data/val/


cp: cannot stat '/content/data/val/scaffold_protbert_E3_val_embeddings.npy': No such file or directory
